In [3]:
import requests
import pandas as pd

# === CONFIGURATION ===
REPOS = [
    "fuzzygroup/polly",
    "fuzzygroup/policounter"
]

# Optional: GitHub token for higher rate limits
GITHUB_TOKEN = None  # e.g., "ghp_xxx" or set via env var

HEADERS = {
    "Accept": "application/vnd.github+json",
}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

def fetch_issues(repo):
    url = f"https://api.github.com/repos/{repo}/issues?state=open&per_page=100"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    issues = response.json()

    # Filter out pull requests
    issues = [i for i in issues if "pull_request" not in i]

    # Flatten data
    rows = []
    for i in issues:
        rows.append({
            "repo": repo,
            "id": i["number"],
            "title": i["title"],
            "assignees": [a["login"] for a in i["assignees"]],
            "labels": [label["name"] for label in i["labels"]],
            "state": i["state"],
            "created_at": i["created_at"],
            "url": i["html_url"]
        })

    return pd.DataFrame(rows)

# === MAIN ===
all_dfs = []
for repo in REPOS:
    print(f"Fetching issues from: {repo}")
    df = fetch_issues(repo)
    all_dfs.append(df)
    # Optionally save per repo
    df.to_csv(f"{repo.replace('/', '_')}_issues.csv", index=False)

# Combine all into one dataframe (optional)
combined_df = pd.concat(all_dfs, ignore_index=True)

Fetching issues from: fuzzygroup/polly
Fetching issues from: fuzzygroup/policounter


In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)  

# Loop through each repo’s dataframe and display
for df in all_dfs:
    if df.empty:
        continue

    # Filter for issues with no assignees or assignee includes "tetrismegistus"
    filtered = df[df["assignees"].apply(lambda a: not a or "tetrismegistus" in a)]

    if filtered.empty:
        continue

    repo_name = filtered.repo.iloc[0]
    print(f"\n📌 Filtered Issues for: {repo_name}")
    display(filtered)



📌 Filtered Issues for: fuzzygroup/polly


,repo,id,title,assignees,labels,state,created_at,url
0,fuzzygroup/polly,78,Default the event state to be the current state where the user comes from based on a geocoding call,[tetrismegistus],[],open,2025-07-15T16:44:41Z,https://github.com/fuzzygroup/polly/issues/78
1,fuzzygroup/polly,76,Global code refactor as a learning tool for aric,[tetrismegistus],[enhancement],open,2025-07-03T08:57:08Z,https://github.com/fuzzygroup/polly/issues/76
2,fuzzygroup/polly,73,Prevent the event attendance / event volunteer / event speaker application buttons from showing up IF THE USER ALREADY DID THEM,[tetrismegistus],[],open,2025-06-26T09:13:46Z,https://github.com/fuzzygroup/polly/issues/73
3,fuzzygroup/polly,72,I want you to write us a rake task which finds problems in our forms,[tetrismegistus],[],open,2025-06-26T08:58:16Z,https://github.com/fuzzygroup/polly/issues/72
4,fuzzygroup/polly,71,JavaScript capture to cookie in the event of a crash on editing text fields,[tetrismegistus],[],open,2025-06-25T08:50:48Z,https://github.com/fuzzygroup/polly/issues/71
5,fuzzygroup/polly,70,Work with Alisa to produce a better looking (yes that's fucking nebulous) Event Attendance form,[tetrismegistus],[],open,2025-06-25T06:48:50Z,https://github.com/fuzzygroup/polly/issues/70
6,fuzzygroup/polly,69,Get Merit Gem working and Get past the Observers issue,[tetrismegistus],[],open,2025-06-24T08:03:00Z,https://github.com/fuzzygroup/polly/issues/69
7,fuzzygroup/polly,68,Integrate Rails Admin into Polly in a Branch,[tetrismegistus],[],open,2025-06-24T07:55:39Z,https://github.com/fuzzygroup/polly/issues/68
8,fuzzygroup/polly,67,Figure out how people can add their own profile images to Polly,[tetrismegistus],[],open,2025-06-24T07:50:04Z,https://github.com/fuzzygroup/polly/issues/67
10,fuzzygroup/polly,64,Protest Alert Feature,[tetrismegistus],[],open,2025-06-23T17:24:24Z,https://github.com/fuzzygroup/polly/issues/64



📌 Filtered Issues for: fuzzygroup/policounter


,repo,id,title,assignees,labels,state,created_at,url
0,fuzzygroup/policounter,35,change readme for project,[],[],open,2025-07-08T18:46:15Z,https://github.com/fuzzygroup/policounter/issues/35
1,fuzzygroup/policounter,34,switch to post from get,[],[],open,2025-07-08T18:14:13Z,https://github.com/fuzzygroup/policounter/issues/34
2,fuzzygroup/policounter,33,lock up communication to polly,[],[],open,2025-07-08T18:13:15Z,https://github.com/fuzzygroup/policounter/issues/33
3,fuzzygroup/policounter,32,expose event fields to the UI,[],[enhancement],open,2025-05-09T05:12:09Z,https://github.com/fuzzygroup/policounter/issues/32
4,fuzzygroup/policounter,31,count should be integer,[tetrismegistus],[bug],open,2025-05-09T04:02:32Z,https://github.com/fuzzygroup/policounter/issues/31
5,fuzzygroup/policounter,29,expose observation fields to UI,[tetrismegistus],[enhancement],open,2025-05-09T01:53:01Z,https://github.com/fuzzygroup/policounter/issues/29
6,fuzzygroup/policounter,26,Get Policounter UI cleaned up -- OVERALL TICKET,[tetrismegistus],[],open,2025-04-28T13:20:09Z,https://github.com/fuzzygroup/policounter/issues/26
7,fuzzygroup/policounter,24,Write an essay about HOW TO MEASURE CROWDS for grass roots political activity,[tetrismegistus],[documentation],open,2025-04-25T17:48:25Z,https://github.com/fuzzygroup/policounter/issues/24
8,fuzzygroup/policounter,23,DB backup,[tetrismegistus],[enhancement],open,2025-04-25T17:47:37Z,https://github.com/fuzzygroup/policounter/issues/23
9,fuzzygroup/policounter,21,rebuild the architecture around a sharing model (using a share of 1 of 3 types per whiteboard discussion),[tetrismegistus],[enhancement],open,2025-04-25T17:46:20Z,https://github.com/fuzzygroup/policounter/issues/21


In [3]:
df.save("issues.csv")


NameError: name 'df' is not defined